In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

In [2]:
data = pd.read_csv('../../data/train.csv')
subset = ['male', 'female', 'homosexual_gay_or_lesbian', 'christian', 'jewish']
subset += ['muslim', 'black', 'white', 'psychiatric_or_mental_illness']

In [3]:
weights = data[['id','target'] + subset].copy()
weights['base'] = 1
weights['normal'] = (weights[subset].fillna(0).values>=0.5).sum(axis=1).astype(bool).astype(int)
weights['group_1'] = (weights['target'].values>=0.5).astype(bool).astype(np.int)
weights['group_1'] += (weights[subset].fillna(0).values < 0.5).sum(axis=1).astype(bool).astype(int)
weights['group_1'] = (weights['group_1'] > 1).astype(bool).astype(int)
weights['group_2'] = (weights['target'].values<0.5).astype(bool).astype(np.int)
weights['group_2'] += (weights[subset].fillna(0).values >= 0.5).sum(axis=1).astype(bool).astype(int)
weights['group_2'] = (weights['group_2'] > 1).astype(bool).astype(int)
weights = weights[['id', 'base', 'normal','group_1','group_2']]
weights['weight'] = weights['base'] + weights['normal'] + weights['group_1'] + weights['group_2']
weights = weights[['id','weight']]
weights.to_csv('../../data/weights.csv', index=False)
weights['weight'].value_counts()

1    1523952
3     163810
2     117112
Name: weight, dtype: int64

In [4]:
FEATURES = ['id','comment_text','weight']
LABELS = ['target'] 
AUX = ['severe_toxicity', 'obscene', 'identity_attack', 'insult', 'threat']

data = pd.read_csv('../../data/train.csv')
weights = pd.read_csv('../../data/weights.csv', usecols=['id','weight'])
data = weights.merge(data, on='id')
data = data[FEATURES + LABELS + AUX]
data[LABELS] = data[LABELS].fillna(0.)
data[AUX] = data[AUX].fillna(0.)
data['comment_text'] = data['comment_text'].fillna('none blank')

split = StratifiedKFold(n_splits=5, random_state=2017)
idx = 0
for train_idx, valid_idx in split.split(data.index, data.weight):
    idx += 1
    train = data.iloc[train_idx, :]
    valid = data.iloc[valid_idx, :]
    train.to_csv('../../data/train_data_{}.csv'.format(idx), index=False)
    valid.to_csv('../../data/valid_data_{}.csv'.format(idx), index=False)
    print(train.shape, valid.shape)

(1443898, 9) (360976, 9)
(1443898, 9) (360976, 9)
(1443900, 9) (360974, 9)
(1443900, 9) (360974, 9)
(1443900, 9) (360974, 9)
